In [ ]:
!pip install torch rasterio geopandas numpy tqdm fiona

In [ ]:
!pip install -U segmentation-models-pytorch timm

In [ ]:
import os
import numpy as np
import torch
import rasterio
from rasterio.windows import Window
from rasterio.features import rasterize
import geopandas as gpd
from tqdm import tqdm
from shapely.geometry import box
import pandas as pd
import fiona
# GPU setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
image_paths = [
    "../Dataset/28996_NADALA_ORTHO.tif",
    "../Dataset/37458_fattu_bhila_ortho_3857.tif",
    "../Dataset/37774_bagga ortho_3857.tif",
    "../Dataset/PINDORI MAYA SINGH-TUGALWAL_28456_ortho.tif",
    "../Dataset/TIMMOWAL_37695_ORI.tif"
]

gpkg_path = "../Dataset/nadalashp.gpkg"
output_dir="processed_patches"
os.makedirs("processed_patches/images", exist_ok=True)
os.makedirs("processed_patches/masks", exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

In [ ]:
layers = fiona.listlayers(gpkg_path)
print("Layers:", layers)

In [ ]:
import fiona
import pandas as pd

layers = fiona.listlayers(gpkg_path)
print("Layers:", layers)

gdfs = []

TARGET_CRS = "EPSG:3857"

for layer in layers:
    gdf_layer = gpd.read_file(gpkg_path, layer=layer)

    print(f"{layer} CRS:", gdf_layer.crs)

    if gdf_layer.crs != TARGET_CRS:
        gdf_layer = gdf_layer.to_crs(TARGET_CRS)

    gdf_layer["feature_type"] = layer
    gdfs.append(gdf_layer)

# Merge
gdf = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True), crs=TARGET_CRS)

print("Total features:", len(gdf))
print("Classes:", gdf["feature_type"].unique())
print("Final CRS:", gdf.crs)


In [ ]:
# Clean invalid geometries
print("Before cleaning:", len(gdf))

gdf = gdf[~gdf.geometry.is_empty]

gdf["geometry"] = gdf.geometry.apply(
    lambda geom: geom.buffer(0) if geom.geom_type not in ["LineString", "MultiLineString", "Point"] else geom
)

gdf = gdf[gdf.is_valid]

print("After cleaning:", len(gdf))

In [ ]:
# CLASS MERGING

def map_class(feature):
    if feature == "built_up_area_typ":
        return "Building"
    
    elif feature in ["road", "road_centre_line", "bridge"]:
        return "Road"
    
    elif feature in ["water_body", "water_body_line", "waterbody_point"]:
        return "Waterbody"
    
    elif feature in ["utility", "utility_poly_"]:
        return "Utility"
    
    elif feature == "railway":
        return None   # ignore railway
    
    else:
        return None


gdf["final_class"] = gdf["feature_type"].apply(map_class)

#Remove unwanted
gdf = gdf[gdf["final_class"].notnull()].copy()


# FIXED CLASS ORDER
CLASS_NAMES = [
    "Background",
    "Building",
    "Road",
    "Waterbody",
    "Utility"
]

class_map = {name: idx for idx, name in enumerate(CLASS_NAMES)}
NUM_CLASSES = len(CLASS_NAMES)

print("Class Map:", class_map)
print("Remaining Classes:", gdf["final_class"].unique())

In [ ]:
patch_size = 256
counter = 0

output_img_dir = "processed_patches/images"
output_mask_dir = "processed_patches/masks"

os.makedirs(output_img_dir, exist_ok=True)
os.makedirs(output_mask_dir, exist_ok=True)

for image_path in image_paths:

    print(f"\nProcessing: {image_path}")

    with rasterio.open(image_path) as src:

        gdf_local = gdf.to_crs(src.crs)

        for i in tqdm(range(0, src.height, patch_size)):
            for j in range(0, src.width, patch_size):

                window = Window(j, i, patch_size, patch_size)
                img = src.read(window=window)

                if img.shape[1] != patch_size or img.shape[2] != patch_size:
                    continue

                transform = src.window_transform(window)
                bounds = rasterio.windows.bounds(window, src.transform)

                gdf_patch = gdf_local.cx[bounds[0]:bounds[2], bounds[1]:bounds[3]].copy()

                if gdf_patch.empty:
                    continue

                # PRIORITY (thin features LAST → so reverse later)
                priority_order = [
                    "built_up_area_typ",
                    "road",
                    "water_body",
                    "utility_poly_",
                    "waterbody_point",
                    "bridge",
                    "utility",
                    "water_body_line",
                    "railway",
                    "road_centre_line"
                ]

                gdf_patch["priority"] = gdf_patch["feature_type"].apply(
                    lambda x: priority_order.index(x) if x in priority_order else 999
                )

                # reverse order so thin features overwrite
                gdf_patch = gdf_patch.sort_values("priority")

                #BUFFER THIN FEATURES
                def buffer_geom(row):
                    geom = row.geometry
                    if row["final_class"] == "Waterbody":
                        if geom.geom_type in ["LineString", "MultiLineString"]:
                            return geom.buffer(4)   #  bigger buffer
                        if geom.geom_type == "Point":
                            return geom.buffer(5)


                    if geom.geom_type in ["LineString", "MultiLineString"]:
                        return geom.buffer(3)

                    if geom.geom_type == "Point":
                        return geom.buffer(3)

                    return geom

                gdf_patch["geometry"] = gdf_patch.apply(buffer_geom, axis=1)

                #  FILTER valid classes only
                gdf_patch = gdf_patch[gdf_patch["final_class"].isin(class_map.keys())]

                if gdf_patch.empty:
                    continue

                shapes = [
                    (geom, class_map[val])
                    for geom, val in zip(gdf_patch.geometry, gdf_patch["final_class"])
                    if geom is not None and not geom.is_empty and geom.is_valid
                ]

                if len(shapes) == 0:
                    continue

                mask = rasterize(
                    shapes,
                    out_shape=(patch_size, patch_size),
                    transform=transform,
                    fill=0,
                    dtype='uint8'
                )

                #  BALANCED PATCH FILTER
                total_pixels = patch_size * patch_size
                unique, counts = np.unique(mask, return_counts=True)

                if max(counts) / total_pixels > 0.95:
                    continue

                img = np.transpose(img, (1, 2, 0))[:, :, :3]

                if np.std(img) < 5:
                    continue

                img = img.astype(np.float32)
                img = (img - img.min()) / (img.max() - img.min() + 1e-8)

                np.save(f"{output_img_dir}/img_{counter}.npy", img)
                np.save(f"{output_mask_dir}/mask_{counter}.npy", mask)

                counter += 1

print("Dataset created:", counter)

In [ ]:
patch_size = 256
counter = 0

output_img_dir = "processed_patches/images"
output_mask_dir = "processed_patches/masks"

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import random

# FIXED CLASS COLORS (same as training)
CLASS_NAMES = [
    "Background",
    "Building",
    "Road",
    "Waterbody",
    "Utility"
]

CLASS_COLORS = [
    (0, 0, 0),        # Background
    (255, 0, 0),      # Building
    (0, 255, 0),      # Road
    (0, 0, 255),      # Waterbody
    (255, 255, 0),    # Utility
]

def colorize(mask):
    h, w = mask.shape
    colored = np.zeros((h, w, 3), dtype=np.uint8)
    for i, color in enumerate(CLASS_COLORS):
        colored[mask == i] = color
    return colored

#  Pick random samples from generated dataset (NOT val_ds)
img_files = sorted(os.listdir(output_img_dir))
indices = random.sample(range(len(img_files)), 5)

for idx in indices:

    img = np.load(os.path.join(output_img_dir, img_files[idx]))
    mask = np.load(os.path.join(output_mask_dir, img_files[idx].replace("img_", "mask_")))

    #  Normalize image for display
    img_disp = (img - img.min()) / (img.max() - img.min() + 1e-8)

    mask_color = colorize(mask)

    #  Overlay
    overlay = (0.7 * img_disp + 0.3 * (mask_color / 255.0)).clip(0,1)

    print("Classes in mask:",
          [CLASS_NAMES[i] for i in np.unique(mask)])

    plt.figure(figsize=(15,4))

    plt.subplot(1,3,1)
    plt.title("Image")
    plt.imshow(img_disp)
    plt.axis('off')

    plt.subplot(1,3,2)
    plt.title("Mask")
    plt.imshow(mask_color)
    plt.axis('off')

    plt.subplot(1,3,3)
    plt.title("Overlay (MOST IMPORTANT)")
    plt.imshow(overlay)
    plt.axis('off')

    plt.show()

In [ ]:
from torch.utils.data import Dataset
import os
import numpy as np
import torch
import random

class GeoDataset(Dataset):
    def __init__(self, img_dir, mask_dir):
        self.img_files = sorted(os.listdir(img_dir))
        self.img_dir = img_dir
        self.mask_dir = mask_dir

    def __len__(self):
        return len(self.img_files)

    def __getitem__(self, idx):

        img_name = self.img_files[idx]

        img = np.load(os.path.join(self.img_dir, img_name))
        mask_name = img_name.replace("img_", "mask_")
        mask = np.load(os.path.join(self.mask_dir, mask_name))

        # convert to tensor FIRST
        img = torch.tensor(img, dtype=torch.float32).permute(2, 0, 1)
        mask = torch.tensor(mask, dtype=torch.long)

        #  AUGMENTATION (AFTER loading)
        if random.random() > 0.5:
            img = torch.flip(img, dims=[2])   # width
            mask = torch.flip(mask, dims=[1])

        if random.random() > 0.5:
            img = torch.flip(img, dims=[1])   # height
            mask = torch.flip(mask, dims=[0])

        #  NORMALIZATION (AFTER augmentation)
        img = (img - img.mean()) / (img.std() + 1e-8)

        return img, mask

In [ ]:
from torch.utils.data import DataLoader, random_split
import torch

dataset = GeoDataset(output_img_dir, output_mask_dir)

# reproducible split
generator = torch.Generator().manual_seed(42)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_ds, val_ds = random_split(dataset, [train_size, val_size], generator=generator)

#  slightly larger batch 
train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=4, shuffle=False, num_workers=4, pin_memory=True)

print("Train:", len(train_ds), "Val:", len(val_ds))

In [ ]:
import segmentation_models_pytorch as smp

print(smp.__version__)

In [ ]:
import segmentation_models_pytorch as smp
import torch.nn as nn

model = smp.FPN(
    encoder_name="resnet50",   
    encoder_weights="imagenet",
    in_channels=3,
    classes=NUM_CLASSES
).to(device)

In [ ]:
%%time
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm
import segmentation_models_pytorch as smp
import torch.nn as nn

dice_loss = smp.losses.DiceLoss(mode='multiclass',smooth=1.0)
#ce_loss = nn.CrossEntropyLoss(ignore_index=255)
ce_loss = nn.CrossEntropyLoss(weight=torch.tensor([0.5,1,1,1,1]).to(device))
def loss_fn(pred, target):
    return 0.5 * ce_loss(pred, target) + 0.5 * dice_loss(pred, target)

optimizer = optim.Adam(model.parameters(), lr=3e-4)


scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=2, factor=0.5
)

scaler = GradScaler()

EPOCHS = 50

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for imgs, masks in tqdm(train_loader):
        imgs = imgs.to(device)
        masks = masks.long().to(device)

        optimizer.zero_grad()

        with autocast():
            outputs = model(imgs)
            loss = loss_fn(outputs, masks)

        scaler.scale(loss).backward()

        # gradient clipping 
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    scheduler.step(avg_loss)

    print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}")
    if torch.isnan(loss):
        print("NaN detected — stopping training")
        break
            

In [ ]:
torch.save(model.state_dict(), "model_multiclass.pth")

In [ ]:
import numpy as np
import torch

model.eval()

total_correct = 0
total_pixels = 0

#  For IoU accumulation
intersection = np.zeros(NUM_CLASSES)
union = np.zeros(NUM_CLASSES)

with torch.no_grad():
    for imgs, masks in val_loader:

        imgs = imgs.to(device)
        masks = masks.to(device)

        outputs = model(imgs)
        preds = torch.argmax(outputs, dim=1)

        #  Pixel Accuracy
        total_correct += (preds == masks).sum().item()
        total_pixels += masks.numel()

        preds_np = preds.cpu().numpy()
        masks_np = masks.cpu().numpy()

        #  Accumulate IoU per class
        for cls in range(1, NUM_CLASSES):  # skip background
            pred_cls = (preds_np == cls)
            mask_cls = (masks_np == cls)

            inter = (pred_cls & mask_cls).sum()
            uni = (pred_cls | mask_cls).sum()

            intersection[cls] += inter
            union[cls] += uni

#  Final metrics
pixel_acc = total_correct / total_pixels

ious = []
for cls in range(1, NUM_CLASSES):
    if union[cls] == 0:
        continue
    ious.append(intersection[cls] / (union[cls] + 1e-6))

mean_iou = np.mean(ious)

print("Pixel Accuracy:", pixel_acc)
print("Mean IoU:", mean_iou)

#  Save model
torch.save(model.state_dict(), "model_multiclass.pth")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import random
from matplotlib.patches import Patch

model.eval()

# CLASS NAMES
CLASS_NAMES = [
    "Background",
    "Building",
    "Road",
    "Waterbody",
    "Utility"
]

#  COLORS
CLASS_COLORS = [
    (0, 0, 0),
    (255, 0, 0),
    (0, 255, 0),
    (0, 0, 255),
    (255, 255, 0),
]

def colorize(mask):
    h, w = mask.shape
    colored = np.zeros((h, w, 3), dtype=np.uint8)
    for i, color in enumerate(CLASS_COLORS):
        colored[mask == i] = color
    return colored

#  RANDOM 10 samples
indices = random.sample(range(len(val_ds)), 100)

print("Selected indices:", indices)

correct_samples = 0

for idx in indices:

    img, mask = val_ds[idx]
    img_input = img.unsqueeze(0).to(device)

    #  TTA
    with torch.no_grad():
        pred1 = model(img_input)

        img_flip = torch.flip(img_input, dims=[3])
        pred2 = model(img_flip)
        pred2 = torch.flip(pred2, dims=[3])

        outputs = (pred1 + pred2) / 2
        pred = torch.argmax(outputs, dim=1).cpu()[0]

    #  IMAGE
    img_disp = img.permute(1,2,0).cpu().numpy()
    img_disp = (img_disp - img_disp.min()) / (img_disp.max() - img_disp.min() + 1e-8)

    mask = mask.cpu().numpy()
    pred = pred.numpy()

    #  CLASS LISTS (IGNORE BACKGROUND OPTIONAL)
    gt_classes = set(np.unique(mask))
    pred_classes = set(np.unique(pred))

    # REMOVE BACKGROUND (optional but better)
    gt_classes.discard(0)
    pred_classes.discard(0)

    #  CHECK MATCH
    if gt_classes == pred_classes:
        is_correct = 1
    elif len(gt_classes)<=len(pred_classes):
            for i in gt_classes:
                if i not in pred_classes:
                    break
            is_correct=1
    else:
        is_correct=0
    
    if is_correct:
        correct_samples += 1

    print("\nGT classes:", [CLASS_NAMES[i] for i in gt_classes])
    print("Pred classes:", [CLASS_NAMES[i] for i in pred_classes])
    print("Match:", "✅ YES" if is_correct else "❌ NO")

    #  COLORIZE
    mask_color = colorize(mask)
    pred_color = colorize(pred)

    overlay = (0.7 * img_disp + 0.3 * (pred_color / 255.0)).clip(0,1)

    legend_elements = [
        Patch(facecolor=np.array(color)/255.0, label=name)
        for color, name in zip(CLASS_COLORS, CLASS_NAMES)
    ]

    plt.figure(figsize=(14,4))

    plt.subplot(1,4,1)
    plt.title("Image")
    plt.imshow(img_disp)
    plt.axis('off')

    plt.subplot(1,4,2)
    plt.title("Ground Truth")
    plt.imshow(mask_color)
    plt.axis('off')

    plt.subplot(1,4,3)
    plt.title("Prediction")
    plt.imshow(pred_color)
    plt.axis('off')

    plt.subplot(1,4,4)
    plt.title("Overlay")
    plt.imshow(overlay)
    plt.axis('off')

    plt.legend(
        handles=legend_elements,
        bbox_to_anchor=(1.05, 1),
        loc='upper left'
    )

    plt.tight_layout()
    plt.show()

#  FINAL RESULT
print("\n Correct Predictions:", correct_samples, "/ 100")
print(" Accuracy:", correct_samples / 100)

In [ ]:
intersection = np.zeros(NUM_CLASSES)
union = np.zeros(NUM_CLASSES)

model.eval()

with torch.no_grad():
    for imgs, masks in val_loader:

        imgs = imgs.to(device)
        masks = masks.to(device)

        #  TTA
        pred1 = model(imgs)

        imgs_flip = torch.flip(imgs, dims=[3])
        pred2 = model(imgs_flip)
        pred2 = torch.flip(pred2, dims=[3])

        outputs = (pred1 + pred2) / 2

        preds = torch.argmax(outputs, dim=1)

        preds_np = preds.cpu().numpy()
        masks_np = masks.cpu().numpy()

        for cls in range(1, NUM_CLASSES):
            pred_cls = (preds_np == cls)
            mask_cls = (masks_np == cls)

            inter = (pred_cls & mask_cls).sum()
            uni = (pred_cls | mask_cls).sum()

            intersection[cls] += inter
            union[cls] += uni

ious = []
for cls in range(1, NUM_CLASSES):
    if union[cls] > 0:
        ious.append(intersection[cls] / (union[cls] + 1e-6))

print("Mean IoU:", np.mean(ious))

#### Testing

In [ ]:
import segmentation_models_pytorch as smp
import torch

NUM_CLASSES = len(CLASS_NAMES)

#  recreate SAME model architecture
model = smp.FPN(
    encoder_name="resnet50",
    encoder_weights=None,   
    in_channels=3,
    classes=NUM_CLASSES
).to(device)

#  load weights
model.load_state_dict(torch.load("model_multiclass.pth", map_location=device))

#  evaluation mode
model.eval()

print("✅ Model loaded successfully")

In [ ]:
import fiona

layers = fiona.listlayers(gpkg_path)
print("Layers:", layers)


In [ ]:
#  INPUTS
ortho_path = "../Dataset/Test/NAGUL_450171_MADASE_450172_GHOTPAL_450137_ORTHO.tif"
gpkg_path = "../Dataset/Test/kutru_gpkg_test.gpkg"
PATCH_SIZE = 256


In [ ]:
import geopandas as gpd
import pandas as pd
import fiona

#  LOAD GPKG
layers = fiona.listlayers(gpkg_path)

gdfs = []
for layer in layers:
    gdf_layer = gpd.read_file(gpkg_path, layer=layer)
    gdf_layer["feature_type"] = layer
    gdfs.append(gdf_layer)

gdf_test = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True))

#  SAME CLASS MAPPING AS TRAINING
def map_class(feature):
    if feature == "built_up_area_type":
        return "Building"
    elif feature in ["road", "road_centre_line", "bridge"]:
        return "Road"
    elif feature in ["water_body", "water_body_line", "waterbody_point"]:
        return "Waterbody"
    elif feature in ["utility", "utility_poly_"]:
        return "Utility"
    else:
        return None   # ignore others

gdf_test["final_class"] = gdf_test["feature_type"].apply(map_class)

#  REMOVE unwanted
gdf_test = gdf_test[gdf_test["final_class"].notnull()].copy()

print(" gdf_test ready")
print("Classes:", gdf_test["final_class"].unique())

In [ ]:
import os
import numpy as np
import rasterio
from rasterio.windows import Window
from rasterio.features import rasterize
from tqdm import tqdm

PATCH_SIZE = 256
MAX_SAMPLES = 200   #  limit here

img_dir = "test_patches/images"
mask_dir = "test_patches/masks"

os.makedirs(img_dir, exist_ok=True)
os.makedirs(mask_dir, exist_ok=True)

counter = 0

with rasterio.open(ortho_path) as src:

    gdf_local = gdf_test.to_crs(src.crs)

    minx, miny, maxx, maxy = gdf_local.total_bounds

    row_min, col_min = src.index(minx, maxy)
    row_max, col_max = src.index(maxx, miny)

    row_min, row_max = sorted([row_min, row_max])
    col_min, col_max = sorted([col_min, col_max])

    for i in tqdm(range(row_min, row_max, PATCH_SIZE)):

        #  early stop outer loop
        if counter >= MAX_SAMPLES:
            break

        for j in range(col_min, col_max, PATCH_SIZE):

            #  early stop inner loop
            if counter >= MAX_SAMPLES:
                break

            window = Window(j, i, PATCH_SIZE, PATCH_SIZE)
            img = src.read(window=window)

            if img.shape[1] != PATCH_SIZE or img.shape[2] != PATCH_SIZE:
                continue

            img = np.transpose(img, (1,2,0))[:, :, :3]

            if np.mean(img) > 240:
                continue

            transform = src.window_transform(window)
            bounds = rasterio.windows.bounds(window, src.transform)

            gdf_patch = gdf_local.cx[bounds[0]:bounds[2], bounds[1]:bounds[3]]

            if gdf_patch.empty:
                continue

            #  GT mask
            shapes = [
                (geom, class_map[val])
                for geom, val in zip(gdf_patch.geometry, gdf_patch["final_class"])
                if val in class_map
            ]

            if len(shapes) == 0:
                continue

            mask = rasterize(
                shapes,
                out_shape=(PATCH_SIZE, PATCH_SIZE),
                transform=transform,
                fill=255,
                dtype='uint8'
            )

            #  normalize image
            img = img.astype(np.float32)
            img = (img - img.min()) / (img.max() - img.min() + 1e-8)

            np.save(f"{img_dir}/img_{counter}.npy", img)
            np.save(f"{mask_dir}/mask_{counter}.npy", mask)

            counter += 1

print("Test dataset created:", counter)

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torch
import numpy as np
import os

class TestDataset(Dataset):
    def __init__(self, img_dir, mask_dir):
        self.files = sorted(os.listdir(img_dir))
        self.img_dir = img_dir
        self.mask_dir = mask_dir

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img_name = self.files[idx]

        img = np.load(os.path.join(self.img_dir, img_name))
        mask = np.load(os.path.join(self.mask_dir, img_name.replace("img_", "mask_")))

        img = torch.tensor(img, dtype=torch.float32).permute(2,0,1)
        mask = torch.tensor(mask, dtype=torch.long)

        return img, mask

test_ds = TestDataset(img_dir, mask_dir)
test_loader = DataLoader(test_ds, batch_size=4, shuffle=False)

print("Test samples:", len(test_ds))

In [ ]:
import random
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch

#  number of samples
NUM_SAMPLES = 30

CLASS_NAMES = ["Background","Building","Road","Waterbody","Utility"]
CLASS_COLORS = [(0,0,0),(255,0,0),(0,255,0),(0,0,255),(255,255,0)]

def colorize(mask):
    h, w = mask.shape
    out = np.zeros((h,w,3), dtype=np.uint8)
    for i,c in enumerate(CLASS_COLORS):
        out[mask==i] = c
    return out

#  safe sampling
indices = random.sample(range(len(test_ds)), min(NUM_SAMPLES, len(test_ds)))
total = len(indices)

correct = 0

model.eval()

for idx in indices:

    img, mask = test_ds[idx]
    img_input = img.unsqueeze(0).to(device)

    #  TTA
    with torch.no_grad():
        pred1 = model(img_input)

        img_flip = torch.flip(img_input, dims=[3])
        pred2 = model(img_flip)
        pred2 = torch.flip(pred2, dims=[3])

        outputs = (pred1 + pred2) / 2
        pred = torch.argmax(outputs, dim=1).cpu()[0]

    #  prepare display
    img_disp = img.permute(1,2,0).numpy()
    img_disp = (img_disp - img_disp.min()) / (img_disp.max() + 1e-8)

    mask = mask.numpy()
    pred = pred.numpy()

    # SAFE class filtering (FIXED)
    gt_classes = set(i for i in np.unique(mask) if i < len(CLASS_NAMES))
    pred_classes = set(i for i in np.unique(pred) if i < len(CLASS_NAMES))

    # remove background
    gt_classes.discard(0)
    pred_classes.discard(0)

    #  match logic
    is_correct = gt_classes.issubset(pred_classes)

    if is_correct:
        correct += 1

    print("\nGT:", [CLASS_NAMES[i] for i in gt_classes])
    print("Pred:", [CLASS_NAMES[i] for i in pred_classes])
    print("Match:", "✅ YES" if is_correct else "❌ NO")

    #  visualization
    mask_vis = np.where(mask == 255, 0, mask)  # remove ignore visually

    mask_color = colorize(mask_vis)
    pred_color = colorize(pred)

    overlay = (0.7 * img_disp + 0.3 * (pred_color / 255.0)).clip(0,1)

    legend_elements = [
        Patch(facecolor=np.array(color)/255.0, label=name)
        for color, name in zip(CLASS_COLORS, CLASS_NAMES)
    ]

    plt.figure(figsize=(14,4))

    plt.subplot(1,4,1)
    plt.title("Image")
    plt.imshow(img_disp)
    plt.axis('off')

    plt.subplot(1,4,2)
    plt.title("Ground Truth")
    plt.imshow(mask_color)
    plt.axis('off')

    plt.subplot(1,4,3)
    plt.title("Prediction")
    plt.imshow(pred_color)
    plt.axis('off')

    plt.subplot(1,4,4)
    plt.title("Overlay")
    plt.imshow(overlay)
    plt.axis('off')

    plt.legend(handles=legend_elements, bbox_to_anchor=(1.05, 1), loc='upper left')

    plt.tight_layout()
    plt.show()

#  FINAL RESULT
print("\n Correct Predictions:", correct, "/", total)
print(" Accuracy:", correct / total)

In [ ]:
%%time
import rasterio
from rasterio.windows import Window
import numpy as np
import torch
import cv2

PATCH_SIZE = 256
BATCH_SIZE = 16   

output_tif = "prediction_clean_gpu.tif"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)
model.eval()

#  mixed precision (fast)
use_amp = True

with rasterio.open(ortho_path) as src:

    profile = src.profile
    profile.update(dtype=rasterio.uint8, count=1, compress='lzw')

    with rasterio.open(output_tif, 'w', **profile) as dst:

        gdf_local = gdf_test.to_crs(src.crs)

        minx, miny, maxx, maxy = gdf_local.total_bounds

        row_min, col_min = src.index(minx, maxy)
        row_max, col_max = src.index(maxx, miny)

        row_min, row_max = sorted([row_min, row_max])
        col_min, col_max = sorted([col_min, col_max])

        batch_imgs = []
        batch_windows = []

        for i in range(row_min, row_max, PATCH_SIZE):
            for j in range(col_min, col_max, PATCH_SIZE):

                window = Window(j, i, PATCH_SIZE, PATCH_SIZE)

                bounds = rasterio.windows.bounds(window, src.transform)
                gdf_patch = gdf_local.cx[bounds[0]:bounds[2], bounds[1]:bounds[3]]

                if gdf_patch.empty:
                    continue

                img = src.read(window=window)

                if img.shape[1] != PATCH_SIZE or img.shape[2] != PATCH_SIZE:
                    continue

                img = np.transpose(img, (1,2,0))[:, :, :3]

                if np.mean(img) > 240:
                    continue

                #  normalize (same as training)
                img = img.astype(np.float32)
                img = (img - img.min()) / (img.max() - img.min() + 1e-8)

                img_tensor = torch.from_numpy(img).permute(2,0,1)

                batch_imgs.append(img_tensor)
                batch_windows.append(window)

                #  run batch
                if len(batch_imgs) == BATCH_SIZE:

                    batch = torch.stack(batch_imgs).to(device)

                    with torch.no_grad():
                        if use_amp:
                            with torch.cuda.amp.autocast():
                                output = model(batch)
                        else:
                            output = model(batch)

                        preds = torch.argmax(output, dim=1).cpu().numpy()

                    for k, pred in enumerate(preds):
                        pred = cv2.medianBlur(pred.astype(np.uint8), 3)
                        dst.write(pred, 1, window=batch_windows[k])

                    batch_imgs = []
                    batch_windows = []

        #  process remaining
        if len(batch_imgs) > 0:

            batch = torch.stack(batch_imgs).to(device)

            with torch.no_grad():
                if use_amp:
                    with torch.cuda.amp.autocast():
                        output = model(batch)
                else:
                    output = model(batch)

                preds = torch.argmax(output, dim=1).cpu().numpy()

            for k, pred in enumerate(preds):
                pred = cv2.medianBlur(pred.astype(np.uint8), 3)
                dst.write(pred, 1, window=batch_windows[k])

print(" Tif file generated...")

In [ ]:
from rasterio.shutil import copy as rio_copy

rio_copy(
    "prediction_clean_gpu.tif",
    "prediction_clean_gpu_cog.tif",
    driver='COG',
    compress='LZW',
    blocksize=512
)
print("COG generated...")